# Crypto pairs trading (AMA4370)

**Strategy in one sentence:** find two coins whose prices move together, and when the gap between them
(the *spread*) is unusually large, bet that it closes again: buy the cheap one, short the expensive one.

Steps of this notebook:
1. Data: hourly prices and funding rates of 12 Binance perpetual futures, 2023-01 to 2026-10
2. Pair selection: cointegration test of all 66 pairs (**train data only**)
3. Spread and z-score of the chosen pair
4. Calibration: choose the parameters (**train data only**)
5. Results on train (in-sample)
6. Results on test (out-of-sample, looked at **once**, nothing changed afterwards)
7. Conclusions

All code is in `src/data.py` (loading data) and `src/strategy.py` (everything else). All settings are in `config.yaml`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import data, strategy

pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams["figure.figsize"] = (11, 4)
cfg = data.load_config()

## 1. Data

In [ ]:
prices, funding = data.load_data(cfg)       # one column per coin, one row per hour
train_prices, test_prices = data.split(prices, cfg)
print(f"Train: {train_prices.index[0]:%Y-%m-%d} to {train_prices.index[-1]:%Y-%m-%d} ({len(train_prices)} hours)")
print(f"Test:  {test_prices.index[0]:%Y-%m-%d} to {test_prices.index[-1]:%Y-%m-%d} ({len(test_prices)} hours)")

(prices / prices.iloc[0] * 100).plot(logy=True, legend=False, alpha=0.7, title="All 12 coins, start = 100 (log scale)")
plt.axvline(pd.Timestamp(cfg["train_end"], tz="UTC"), color="k", ls="--", label="end of train")
plt.legend(); plt.show()

The data is complete: no missing hours for any coin. Funding is paid every 8 hours. A positive funding rate
means longs pay shorts. In a pair trade we are long one coin and short the other, so the two funding payments
mostly cancel out.

## 2. Pair selection (train data only)

For every pair we run the **Engle-Granger test**: regress log(A) on log(B), then test whether the residual
(= the spread) is stationary (ADF test). A low p-value means the spread tends to come back to its mean.

How to read the table:
- `p_bonferroni`: we test 66 pairs, so about 3 of them look significant at 5% by pure luck. Bonferroni multiplies the p-value by 66 to correct for this.
- `adf_p_a`, `adf_p_b`: is each coin **on its own** already stationary? It should not be (high p-value). If it is, the "pair" is fake.
- `half_life_h`: hours until the spread has closed half of its distance to the mean.

In [ ]:
ranking = strategy.rank_pairs(train_prices)
ranking.head(10)

**What we see:**
- The two best pairs both contain **LTC**. But `adf_p` of LTC alone is about 0.01: LTC moved sideways in a range
  on its own, so these "pairs" are just LTC. We drop them.
- **No pair is significant after Bonferroni** (all `p_bonferroni` far above 0.05). Statistically, no pair is clearly cointegrated over the full train period.
- Half-lives are long (weeks), so we use a **rolling** hedge ratio and z-score that adapt over time.

So the test result alone does not give us a clear winner. We therefore choose the pair with the calibration in section 4:
we take a few liquid, economically related candidates and backtest them on train.
Our choice: **ETH / BNB**. Both are the native coins of the two biggest smart-contract platforms
(Ethereum and BNB Chain), their returns have a correlation of 0.71, and neither is stationary alone.

## 3. Spread and z-score of the chosen pair

- **beta** (hedge ratio): slope of log(ETH) on log(BNB) over the last `hedge_window` hours (rolling, past data only)
- **spread** = log(ETH) − beta · log(BNB) − alpha
- **z-score** = (spread − rolling mean) / rolling std, over the last `zscore_lookback` hours

In [ ]:
res_train = strategy.backtest(prices, funding, cfg, "train")
bars = res_train["bars"]
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
bars["spread"].plot(ax=ax[0], title=f"Spread {cfg['pair'][0]}/{cfg['pair'][1]} (train)")
bars["z"].plot(ax=ax[1], title="z-score with entry (±entry_z) and stop (±stop_z) levels", lw=0.6)
for level, color in [(cfg["entry_z"], "g"), (cfg["stop_z"], "r")]:
    ax[1].axhline(level, color=color, ls="--"); ax[1].axhline(-level, color=color, ls="--")
plt.tight_layout(); plt.show()

## 4. Calibration (train data only)

**Trading rules** (in `strategy.positions`):
- **Entry:** z < −entry_z → long the spread (buy ETH, short BNB). z > +entry_z → short the spread.
- **Take profit:** close when z is back at the mean (exit_z = 0).
- **Stop loss:** close when |z| > stop_z. The spread keeps running away, so maybe the relationship broke.
- **Time stop:** close after time_stop_hours. The spread is not coming back in a reasonable time.
- After a stop we wait until |z| < entry_z before trading again.

**Costs:** 0.05% fee + 0.02% slippage per leg and trade, so one round trip costs about 0.14% of capital.

### 4a. Pair and signal parameters
Sharpe ratio on train for a grid of candidate pairs × hedge window × z-score lookback × entry threshold.

In [ ]:
candidates = [["ETH", "BTC"], ["ETH", "BNB"], ["SOL", "LINK"], ["BTC", "BNB"], ["ADA", "DOGE"], ["LINK", "DOGE"]]
rows = []
for pair, hw, lb, ez in itertools.product(candidates, [480, 720, 1080], [168, 336, 504], [2.0, 2.5, 3.0]):
    m = strategy.backtest(prices, funding, dict(cfg, pair=pair, hedge_window=hw, zscore_lookback=lb, entry_z=ez))["metrics"]
    rows.append({"pair": "/".join(pair), "hedge_window": hw, "lookback": lb, "entry_z": ez,
                 "sharpe": m["sharpe"], "n_trades": m["n_trades"]})
grid = pd.DataFrame(rows)
grid.groupby("pair")["sharpe"].agg(["mean", "max"]).sort_values("max", ascending=False)

In [ ]:
# Sharpe for ETH/BNB: rows = (hedge window, lookback), columns = entry_z
grid[grid.pair == "ETH/BNB"].pivot_table(index=["hedge_window", "lookback"], columns="entry_z", values="sharpe")

**What we see:** most combinations lose money on train, for every pair. Short lookbacks (168 h) are bad
almost everywhere, because the spread then hardly moves away from its own short-term mean and costs dominate.
ETH/BNB has a good region around a hedge window of 720 h (30 days) and a lookback of 336 h (2 weeks), Sharpe ≈ 0.7–0.8.
ADA/DOGE has a similar best cell, but drawdowns of about −30% (ETH/BNB: −17%).

**But the good cells are isolated:** their neighbours are much worse. That is a warning sign: the best cell of
many tries is partly luck (overfitting), so we expect the test result to be weaker than the train result.

**Choice:** ETH/BNB (clear economic link, lower drawdown), hedge_window = 720, zscore_lookback = 336,
entry_z = 2.5 (saved in `config.yaml`).

### 4b. Risk rules

In [ ]:
variants = {"no stops": dict(stop_z=99, time_stop_hours=10**6),
            "stop loss only": dict(time_stop_hours=10**6),
            "time stop only": dict(stop_z=99),
            "both (our choice)": {}}
pd.DataFrame({name: strategy.backtest(prices, funding, dict(cfg, **v))["metrics"] for name, v in variants.items()}
             ).loc[["sharpe", "total_return", "max_drawdown", "n_trades", "win_rate"]]

The **stop loss** at |z| > 3.5 is the rule that matters: it cuts the max drawdown from about −27% to about −16%
and even raises the Sharpe.

The **time stop** of 168 h (one week) does **not** help on train: Sharpe 1.10 without it vs 0.82 with it.
We kept it as a safety rule (no trade can block the capital for weeks). Looking back, a strict train-data
decision would have dropped it. We did **not** change it after seeing the test result, so the test stays clean.

## 5. Results on train (in-sample)

In [ ]:
def show(res):
    b = res["bars"]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    b["equity"].plot(ax=ax[0], title=f"Equity ({res['period']}), start = 1")
    (b["equity"] / b["equity"].cummax().clip(lower=1) - 1).plot(ax=ax[1], title="Drawdown", color="r")
    plt.tight_layout(); plt.show()
    print(res["trades"]["exit_reason"].value_counts().to_string())
    return pd.Series(res["metrics"]).to_frame(res["period"])

show(res_train)

In [ ]:
res_train["trades"].head(10)   # first trades: when, which direction, z at entry/exit, return, why closed

## 6. Results on test (out-of-sample, run once)
Same code, same parameters. The rolling beta and z-score warm up on the end of the train period
(they only look backwards), so no future data is used.

In [ ]:
res_test = strategy.backtest(prices, funding, cfg, "test")
show(res_test)

In [ ]:
# Robustness: what if costs are higher, or without funding?
checks = {}
for period in ["train", "test"]:
    checks[(period, "base")] = strategy.backtest(prices, funding, cfg, period)["metrics"]
    checks[(period, "fees x2")] = strategy.backtest(prices, funding, dict(cfg, fee=2 * cfg["fee"]), period)["metrics"]
    checks[(period, "no funding")] = strategy.backtest(prices, None, cfg, period)["metrics"]
pd.DataFrame(checks).loc[["sharpe", "total_return", "max_drawdown", "n_trades", "sum_gross", "sum_costs", "sum_funding"]]

## 7. Conclusions

- **Train:** Sharpe ≈ 0.8, +44% over 3 years, max drawdown ≈ −17%, ~100 trades.
- **Test:** Sharpe ≈ 0.3, +2.6% over 10 months, max drawdown ≈ −9%, 26 trades. The strategy is still slightly
  positive, but much weaker than on train. That is what we expected, because the chosen parameters were an isolated peak.
- **Costs matter a lot:** fees and slippage eat about a quarter of the gross profit on train and half of it on test.
  With double fees the test result is zero.
- **Funding hardly matters:** being long one perpetual and short another, the payments mostly cancel out.
- **Main lesson:** crypto pairs are not reliably cointegrated (no pair survives the multiple-testing correction),
  so the edge is small and fragile.

**What we would do next:** test the strategy on many pairs at once (a portfolio, more trades, more reliable statistics),
use walk-forward re-calibration, and use maker orders to cut fees.